In [1]:
import pandas as pd
from typing import Dict, List, Union
import numpy as np

In [3]:
class CSVDataLoader:
    """
    通用结构化CSV数据加载器
    支持特性：
    - 自动检测分隔符
    - 类型自动推断与强制转换
    - 缺失值处理
    - 自定义列处理器
    """
    
    def __init__(self, 
                 filepath: str,
                 na_values: List[str] = ["unknown", "nonexistent", ""],
                 dtype_rules: Dict[str, Union[str, type]] = None,
                 processors: Dict[str, callable] = None):
        """
        参数:
            filepath: 文件路径
            na_values: 视为缺失值的字符串列表
            dtype_rules: 列数据类型规则 {'列名': 'int32'|'float'|'category'|...}
            processors: 列自定义处理器 {'列名': 处理函数}
        """
        self.filepath = filepath
        self.na_values = na_values
        self.dtype_rules = dtype_rules or {}
        self.processors = processors or {}
        self.df = None
        self._inferred_dtypes = None

    def _infer_delimiter(self, sample_lines: int = 5) -> str:
        """自动推断分隔符"""
        with open(self.filepath, 'r') as f:
            lines = [f.readline() for _ in range(sample_lines)]
        
        # 常见分隔符优先级
        for delim in [';', ',', '\t', '|']:
            if all(delim in line for line in lines if line.strip()):
                return delim
        return ','  # 默认逗号分隔

    def load(self) -> pd.DataFrame:
        """加载并初步清洗数据"""
        delim = self._infer_delimiter()
        
        # 首次读取用于类型推断
        self.df = pd.read_csv(
            self.filepath,
            delimiter=delim,
            quotechar='"',
            na_values=self.na_values,
            low_memory=False
        )
        
        # 应用数据类型规则
        if self.dtype_rules:
            for col, dtype in self.dtype_rules.items():
                if col in self.df.columns:
                    if dtype == 'category':
                        self.df[col] = self.df[col].astype('category')
                    else:
                        self.df[col] = pd.to_numeric(self.df[col], errors='coerce').astype(dtype)
        
        # 应用自定义处理器
        for col, processor in self.processors.items():
            if col in self.df.columns:
                self.df[col] = self.df[col].apply(processor)
        
        return self.df

    def clean_numeric(self, cols: List[str], fillna: float = None) -> None:
        """清理数值列异常值"""
        for col in cols:
            if col in self.df.columns:
                # 移除非数字字符
                self.df[col] = pd.to_numeric(
                    self.df[col].astype(str).str.replace(r'[^0-9\.\-]', '', regex=True),
                    errors='coerce'
                )
                if fillna is not None:
                    self.df[col].fillna(fillna, inplace=True)

    def get_summary(self) -> Dict:
        """获取数据统计摘要"""
        if self.df is None:
            self.load()
            
        return {
            'dtypes': self.df.dtypes.to_dict(),
            'missing_values': self.df.isna().sum().to_dict(),
            'numeric_stats': self.df.describe(include=np.number).to_dict() 
                           if self.df.select_dtypes(np.number).columns.any() 
                           else None,
            'categorical_stats': self.df.describe(include=['category', 'object']).to_dict() 
                               if self.df.select_dtypes(['category', 'object']).columns.any() 
                               else None
        }

# 使用示例
if __name__ == "__main__":
    # 定义数据处理规则
    rules = {
        'age': 'int32',
        'duration': 'int32',
        'campaign': 'int32',
        'pdays': 'int32',
        'previous': 'int32',
        'emp.var.rate': 'float32',
        'cons.price.idx': 'float32',
        'y': 'category'  # 目标变量转为分类
    }
    
    # 定义特殊列处理器
    processors = {
        'job': lambda x: x.lower().replace('.', '') if pd.notna(x) else x,
        'education': lambda x: x.split('.')[0] if pd.notna(x) and '.' in x else x
    }
    
    # 初始化加载器
    loader = CSVDataLoader(
        filepath="Bank_Marketing_raw.csv",
        na_values=["unknown", "nonexistent", ""],
        dtype_rules=rules,
        processors=processors
    )
    
    # 加载数据
    df = loader.load()
    
    # 清理数值列
    loader.clean_numeric(
        cols=['age', 'duration', 'campaign', 'pdays', 'previous'],
        fillna=-1
    )
    
    # 查看摘要
    print(loader.get_summary())
    
    # 获取处理后的DataFrame
    clean_df = loader.df
    print(clean_df.head())

{'dtypes': {'age': dtype('int64'), 'job': dtype('O'), 'marital': dtype('O'), 'education': dtype('O'), 'default': dtype('O'), 'housing': dtype('O'), 'loan': dtype('O'), 'contact': dtype('O'), 'month': dtype('O'), 'day_of_week': dtype('O'), 'duration': dtype('int64'), 'campaign': dtype('int64'), 'pdays': dtype('int64'), 'previous': dtype('int64'), 'poutcome': dtype('O'), 'emp.var.rate': dtype('float32'), 'cons.price.idx': dtype('float32'), 'cons.conf.idx': dtype('float64'), 'euribor3m': dtype('float64'), 'nr.employed': dtype('float64'), 'y': CategoricalDtype(categories=['no', 'yes'], ordered=False)}, 'missing_values': {'age': 0, 'job': 330, 'marital': 80, 'education': 1731, 'default': 8597, 'housing': 990, 'loan': 990, 'contact': 0, 'month': 0, 'day_of_week': 0, 'duration': 0, 'campaign': 0, 'pdays': 0, 'previous': 0, 'poutcome': 35563, 'emp.var.rate': 0, 'cons.price.idx': 0, 'cons.conf.idx': 0, 'euribor3m': 0, 'nr.employed': 0, 'y': 0}, 'numeric_stats': {'age': {'count': 41188.0, 'mean'

In [5]:
clean_df.to_csv("Bank_Marketing.csv", index=False)  # 不保存行索引

In [ ]:
import pandas as pd
import json

def enrich_feature_meta(csv_path: str, json_path: str, output_path: str):
    # 读取 CSV 数据
    df = pd.read_csv(csv_path)
    
    # 替换 DataFrame 列名中的 '.' 为 '_'
    df.columns = [col.replace('.', '_') for col in df.columns]

    # 读取 JSON 文件并将 key 中的 '-' 替换为 '_'
    with open(json_path, 'r') as f:
        raw_meta = json.load(f)

    # 替换 JSON 的 key
    original_meta = {k.replace('-', '_'): v for k, v in raw_meta.items()}

    # 构建 enriched metadata
    enriched_meta = {}
    for col in df.columns:
        desc = original_meta.get(col, "")  # 若 JSON 中无该列，desc设为空字符串
        col_data = df[col]

        meta = {
            "description": desc
        }

        # 判断类型
        if col_data.nunique() == len(col_data):  # 每个值都不同
            meta["type"] = "unique_identifier"
            meta["unique_values"] = int(col_data.nunique())
        elif col_data.dtype == object or col_data.dtype.name == "category":
            meta["type"] = "categorical"
            meta["values"] = sorted(col_data.dropna().unique().tolist())
        else:
            meta["type"] = "numerical"

        enriched_meta[col] = meta

    # 保存为新 JSON 文件
    with open(output_path, 'w') as f:
        json.dump(enriched_meta, f, indent=2)

    print(f"✅ 已保存 enriched metadata 至：{output_path}")